# NovaCred Data Governance & Privacy Audit
**Prepared by: Governance Officer**

Hi team! As the Governance Officer for our project, I've put together this notebook to run an algorithmic audit on the `raw_credit_applications.json` dataset. 

Since NovaCred uses machine learning for credit decisions, we are operating a "High-Risk AI System" under the EU AI Act. Before our Data Engineer and Data Scientist get too deep into modeling, we need to prove that our data pipeline actually respects user privacy and meets regulatory standards. 

I've broken this down into three main checks:
1. **Data Quality Audit** (Finding the messy data)
2. **Privacy & Data Minimization** (Securing PII)
3. **Fairness & Bias Testing** (Checking for discrimination)

In [10]:
import pandas as pd
import json
import hashlib
import hmac
import secrets

# Let's load the data and flatten the nested JSON so it's easier to work with in Pandas
# Using '../data/' to tell Python to go up one folder, then into the data folder
file_path = '../data/raw_credit_applications.json' 
with open(file_path, 'r') as file:
    raw_json = json.load(file)

df = pd.json_normalize(raw_json)
print(f"Dataset successfully loaded! We have {len(df)} credit applications to review.")

Dataset successfully loaded! We have 502 credit applications to review.


## Part 1: Automated Data Quality Audit
[cite_start]The EU AI Act (Art. 10) is pretty strict about high-risk systems needing error-free, representative data[cite: 220, 224]. If we train a model on garbage data, we get garbage (and legally non-compliant) credit decisions. Let's see what kind of issues are hiding in here.

In [11]:
print("--- RUNNING DATA QUALITY CHECKS ---\n")

# 1. Check for duplicates (Uniqueness)
duplicate_ids = df.duplicated(subset=['_id'], keep=False).sum()
duplicate_ssns = df.duplicated(subset=['applicant_info.ssn'], keep=False).sum()

print(f"Duplicate Application IDs: {duplicate_ids}")
print(f"Duplicate SSNs across different apps: {duplicate_ssns}")
if duplicate_ssns > 0:
    print(" ⚠ Uh oh, multiple applications are sharing the same SSN. This could be fraud or a bad pipeline error.")

# 2. Check for missing data (Completeness)
print("\nChecking for missing critical fields:")
critical_columns = [
    'applicant_info.ssn', 'applicant_info.gender', 
    'applicant_info.date_of_birth', 'financials.annual_income'
]
for col in critical_columns:
    if col in df.columns:
        missing = df[col].isna().sum()
        print(f" - {col}: {missing} records missing ({(missing / len(df)) * 100:.1f}%)")

# 3. Look at how gender is recorded (Consistency)
print("\nChecking gender encodings:")
if 'applicant_info.gender' in df.columns:
    gender_variations = df['applicant_info.gender'].unique()
    print(f" Found {len(gender_variations)} different ways gender is typed out: {gender_variations}")
    print(" ⚠ We definitely need to standardize this before running our bias metrics.")

# 4. Check for impossible business values (Validity)
print("\nChecking for impossible financial numbers:")
if 'financials.annual_income' in df.columns:
    incomes = pd.to_numeric(df['financials.annual_income'], errors='coerce')
    string_incomes = df['financials.annual_income'].apply(lambda x: isinstance(x, str)).sum()
    print(f" - Incomes saved as text strings instead of numbers: {string_incomes}")
    print(f" - Applications with negative or zero income: {(incomes <= 0).sum()}")

if 'financials.debt_to_income' in df.columns:
    impossible_dti = (pd.to_numeric(df['financials.debt_to_income'], errors='coerce') > 1.0).sum()
    print(f" - Applications with over 100% Debt-to-Income ratio: {impossible_dti}")

--- RUNNING DATA QUALITY CHECKS ---

Duplicate Application IDs: 4
Duplicate SSNs across different apps: 11
 ⚠ Uh oh, multiple applications are sharing the same SSN. This could be fraud or a bad pipeline error.

Checking for missing critical fields:
 - applicant_info.ssn: 5 records missing (1.0%)
 - applicant_info.gender: 1 records missing (0.2%)
 - applicant_info.date_of_birth: 1 records missing (0.2%)
 - financials.annual_income: 5 records missing (1.0%)

Checking gender encodings:
 Found 6 different ways gender is typed out: ['Male' 'M' 'F' 'Female' '' nan]
 ⚠ We definitely need to standardize this before running our bias metrics.

Checking for impossible financial numbers:
 - Incomes saved as text strings instead of numbers: 8
 - Applications with negative or zero income: 1
 - Applications with over 100% Debt-to-Income ratio: 1


## Part 2: Privacy by Design (Pseudonymization & Minimization)
[cite_start]Right now, highly sensitive Personally Identifiable Information (PII) like plain-text SSNs and IP addresses are sitting right in the dataset[cite: 229]. [cite_start]That's a massive GDPR violation (Articles 5 and 32)[cite: 229]. 

To fix this, I'm going to write a few functions to anonymize the data:
* **SSNs:** I'll use a cryptographic HMAC hash with a secret salt. This is way safer than standard hashing because it stops attackers from just guessing SSNs until they find a match.
* **Dates of Birth & ZIPs:** I'll group these into broader buckets (like age ranges and regional ZIPs) so we can still analyze trends without pointing to a specific person.

In [12]:
# Generate a secure key for hashing (Note: In the real world, we'd store this in a secure environment variable!)
HMAC_KEY = secrets.token_bytes(32)

def secure_ssn_hash(ssn):
    """Replaces SSN with a secure cryptographic hash."""
    if pd.isna(ssn) or not isinstance(ssn, str):
        return ssn
    return hmac.new(HMAC_KEY, ssn.encode('utf-8'), hashlib.sha256).hexdigest()

def mask_ip_subnet(ip):
    """Removes the last part of the IP address so we can't track exact devices."""
    if pd.isna(ip) or not isinstance(ip, str): return ip
    return '.'.join(ip.split('.')[:3]) + '.XXX'

def bin_age(dob):
    """Turns specific birthdays into age brackets."""
    try:
        # Assuming current year is 2026 based on our project settings
        age = 2026 - pd.to_datetime(dob).year 
        if age < 25: return "18-24"
        elif age < 35: return "25-34"
        elif age < 45: return "35-44"
        elif age < 55: return "45-54"
        else: return "55+"
    except:
        return "Unknown"

# Let's apply these privacy rules to a clean copy of the dataframe
df_clean = df.copy()

if 'applicant_info.ssn' in df_clean.columns:
    df_clean['crypto_ssn'] = df_clean['applicant_info.ssn'].apply(secure_ssn_hash)

if 'applicant_info.ip_address' in df_clean.columns:
    df_clean['masked_ip'] = df_clean['applicant_info.ip_address'].apply(mask_ip_subnet)

if 'applicant_info.date_of_birth' in df_clean.columns:
    df_clean['age_bracket'] = df_clean['applicant_info.date_of_birth'].apply(bin_age)

if 'applicant_info.zip_code' in df_clean.columns:
    # Keep only the first 3 digits of the zip code
    df_clean['regional_zip'] = df_clean['applicant_info.zip_code'].astype(str).str[:3] + "XX"

# Finally, drop the original plain-text PII columns so they are gone for good
cols_to_drop = ['applicant_info.ssn', 'applicant_info.ip_address', 'applicant_info.date_of_birth']
df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], inplace=True)

print("✅ Privacy transformations complete! Here is what our safe data looks like now:")
df_clean[['applicant_info.full_name', 'crypto_ssn', 'masked_ip', 'age_bracket', 'regional_zip']].head()

✅ Privacy transformations complete! Here is what our safe data looks like now:


C:\Users\prate\AppData\Local\Temp\ipykernel_17732\2359054491.py:19: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  age = 2026 - pd.to_datetime(dob).year


,applicant_info.full_name,crypto_ssn,masked_ip,age_bracket,regional_zip
0,Jerry Smith,8b541a3fc88236e2b47adc7b60c4b4adb57ea472fc16a8...,192.168.48.XXX,25-34,100XX
1,Brandon Walker,0a5c4806de3c63fb04e15ec735b52cc1f81d1c0eb73793...,10.1.102.XXX,25-34,100XX
2,Scott Moore,8a651c7c4ed8c4f874ff6d42b8f64f9e9be5c7b527f798...,10.240.193.XXX,35-44,100XX
3,Thomas Lee,3b18912ac18f1eb11780c7f21689184f917f54b2a24291...,192.168.175.XXX,35-44,100XX
4,Brian Rodriguez,bf3bdc0d5ba13166d79c8feeb822dc0ad7ac9f986d1265...,172.29.125.XXX,25-34,100XX


## Part 3: Algorithmic Fairness & Bias Testing
[cite_start]Now we need to see if NovaCred's historical decisions are discriminatory. We'll use the "Four-Fifths Rule" (Disparate Impact)[cite: 137]. [cite_start]Basically, if the approval rate for a marginalized group is less than 80% of the approval rate for a privileged group, we have a legal and ethical problem[cite: 137]. [cite_start]Let's test Gender, Age, and check if ZIP codes are acting as a sneaky proxy for discrimination[cite: 224].

In [13]:
print("--- RUNNING FAIRNESS & BIAS TESTS ---\n")

# First, clean up those messy gender encodings we found earlier
df_clean['std_gender'] = df_clean['applicant_info.gender'].replace({'M': 'Male', 'F': 'Female'})

def check_disparate_impact(data, feature, privileged, unprivileged, target='decision.loan_approved'):
    """Calculates the Disparate Impact ratio between two groups."""
    priv_rate = data[data[feature] == privileged][target].mean()
    unpriv_rate = data[data[feature] == unprivileged][target].mean()
    
    if pd.isna(priv_rate) or pd.isna(unpriv_rate) or priv_rate == 0:
        return None, priv_rate, unpriv_rate
        
    di_ratio = unpriv_rate / priv_rate
    return di_ratio, priv_rate, unpriv_rate

# 1. Gender Bias
di_gender, priv_m, unpriv_f = check_disparate_impact(df_clean, 'std_gender', 'Male', 'Female')
print("1. GENDER BIAS")
print(f" - Male Approval Rate: {priv_m:.1%}")
print(f" - Female Approval Rate: {unpriv_f:.1%}")
print(f" -> Disparate Impact Ratio: {di_gender:.3f}")
if di_gender < 0.8:
    print(" ⚠ FAILED: Ratio is below 0.80. The algorithm is biased against women.")

# 2. Age Bias (Using our new age brackets)
di_age, priv_mid, unpriv_young = check_disparate_impact(df_clean, 'age_bracket', '35-44', '18-24')
print("\n2. AGE BIAS")
print(f" - Age 35-44 Approval Rate: {priv_mid:.1%}")
print(f" - Age 18-24 Approval Rate: {unpriv_young:.1%}")
print(f" -> Disparate Impact Ratio: {di_age:.3f}")

# 3. Proxy Discrimination Check (ZIP Codes)
print("\n3. PROXY DISCRIMINATION (ZIP Code Analysis)")
# We're checking if the model is just rejecting specific ZIP codes that happen to have a lot of women in them.
zip_analysis = df_clean.groupby('regional_zip').agg(
    total_apps=('decision.loan_approved', 'count'),
    approval_rate=('decision.loan_approved', 'mean'),
    female_ratio=('std_gender', lambda x: (x == 'Female').mean())
).sort_values('total_apps', ascending=False).head()

print(zip_analysis.to_string(formatters={'approval_rate': '{:.1%}'.format, 'female_ratio': '{:.1%}'.format}))
print("\nNote for the team: If we see ZIP codes with high female populations getting disproportionately rejected, the model is using geography as a proxy for gender.")

--- RUNNING FAIRNESS & BIAS TESTS ---

1. GENDER BIAS
 - Male Approval Rate: 65.7%
 - Female Approval Rate: 50.6%
 -> Disparate Impact Ratio: 0.770
 ⚠ FAILED: Ratio is below 0.80. The algorithm is biased against women.

2. AGE BIAS
 - Age 35-44 Approval Rate: 67.1%
 - Age 18-24 Approval Rate: 33.3%
 -> Disparate Impact Ratio: 0.497

3. PROXY DISCRIMINATION (ZIP Code Analysis)
              total_apps approval_rate female_ratio
regional_zip                                       
100XX                252         64.3%        11.1%
902XX                230         51.7%        93.5%
300XX                 18         55.6%        44.4%
XX                     1        100.0%         0.0%
nanXX                  1          0.0%         0.0%

Note for the team: If we see ZIP codes with high female populations getting disproportionately rejected, the model is using geography as a proxy for gender.


## Part 4: Official Governance Mapping & Recommendations
*Based on the code output above, here is how our findings map to actual regulations for our README report.*

### 1. GDPR Compliance Gaps [cite: 144, 229]
* **Art. 5(1)(a) - Lawfulness & Transparency:** The dataset has zero evidence of consent (`consent_timestamp` is missing). We currently have no documented legal basis to process this data. Plus, our bias audit proves the outcomes aren't "fair" (DI Ratio < 0.80 for women).
* **Art. 5(1)(c) - Data Minimization:** We successfully fixed some of this! In the code above, we replaced exact birthdates with age brackets and truncated ZIP and IP addresses to limit unnecessary tracking.
* **Art. 5(1)(f) - Integrity & Confidentiality:** We implemented a secure HMAC-SHA256 hash for the SSNs. This protects a permanent, sensitive identifier from plain-text exposure while still allowing our engineers to catch duplicate applications safely.
* **Art. 17 - Right to Erasure:** Because we are missing processing timestamps, NovaCred currently has no automated way to know when to delete old data to comply with "Right to be Forgotten" requests.

### 2. EU AI Act (High-Risk System) [cite: 229]
* **Art. 10 - Data Quality:** Our audit script caught major errors: negative credit histories, impossible debt-to-income ratios, and fragmented gender encodings. We *must* put input validation blocks on our data ingestion pipeline before training any models. 
* **Art. 14 - Human Oversight:** Every single decision in this dataset looks completely automated. We need to recommend a policy where human reviewers must sign off on algorithmic loan rejections before the customer is notified.